# Options ML Models
Live reference — run all cells to refresh charts from `outputs/` and `models_info.json`.

In [ ]:
import json, math, sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
import warnings; warnings.filterwarnings('ignore')

ROOT = Path('.').resolve()
if ROOT.name == 'docs': ROOT = ROOT.parent
OUTPUTS = ROOT / 'outputs'
MODELS  = ROOT / 'models'

# ── BS helpers (mirrors options_trader.GreeksEstimator) ──
from scipy.stats import norm          # optional: falls back to math.erf below

def _ncdf(x):
    try: return float(norm.cdf(x))
    except: return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def _npdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2 * math.pi)

R = 0.05

def d1d2(S, K, T, sigma):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0: return 0.0, 0.0
    d1 = (math.log(S/K) + (R + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    return d1, d1 - sigma*math.sqrt(T)

def bs_price(is_call, S, K, T, sigma):
    if T <= 0: return max(0, S-K) if is_call else max(0, K-S)
    d1, d2 = d1d2(S, K, T, sigma)
    if is_call: return S*_ncdf(d1) - K*math.exp(-R*T)*_ncdf(d2)
    return K*math.exp(-R*T)*_ncdf(-d2) - S*_ncdf(-d1)

def bs_delta(is_call, S, K, T, sigma):
    if T <= 0: return (1.0 if S>K else 0.0) if is_call else (-1.0 if S<K else 0.0)
    d1, _ = d1d2(S, K, T, sigma)
    return _ncdf(d1) if is_call else _ncdf(d1) - 1.0

def bs_gamma(S, K, T, sigma):
    if T <= 0 or sigma <= 0: return 0.0
    d1, _ = d1d2(S, K, T, sigma)
    return _npdf(d1) / (S * sigma * math.sqrt(T))

def bs_theta(is_call, S, K, T, sigma):
    if T <= 0 or sigma <= 0: return 0.0
    d1, d2 = d1d2(S, K, T, sigma)
    ann = (-(S*_npdf(d1)*sigma)/(2*math.sqrt(T))
           - R*K*math.exp(-R*T)*(_ncdf(d2) if is_call else _ncdf(-d2)))
    return ann / 365.0

def bs_vega(S, K, T, sigma):
    if T <= 0 or sigma <= 0: return 0.0
    d1, _ = d1d2(S, K, T, sigma)
    return S * _npdf(d1) * math.sqrt(T) * 0.01

print('imports OK')

---
## Architecture

In [ ]:
def arch():
    fig = go.Figure()
    fig.update_xaxes(visible=False, range=[0,1])
    fig.update_yaxes(visible=False, range=[0,1])
    BW, BH = 0.30, 0.085
    nodes = [
        (0.50, 0.95, 'Bar Data + VIX',                             '#2980B9'),
        (0.25, 0.78, 'DirectionLSTM  hidden=96 · 21 feat',         '#27AE60'),
        (0.25, 0.62, 'meta-RF gate  per-symbol threshold',          '#27AE60'),
        (0.25, 0.46, 'conf >= 0.65  IVR <= 55  IV/RV <= 1.75',     '#E67E22'),
        (0.75, 0.78, 'VolExpansionLSTM  hidden=64 · 10 feat',      '#8E44AD'),
        (0.75, 0.62, 'vol meta-RF',                                 '#8E44AD'),
        (0.75, 0.46, 'vol_prob >= 0.30  IVR <= 25  IV/RV <= 1.70', '#E67E22'),
        (0.15, 0.25, 'ITM Call/Put  delta~0.68  28 DTE',            '#C0392B'),
        (0.85, 0.25, 'ATM Straddle  30 DTE',                        '#C0392B'),
        (0.50, 0.08, 'No Trade',                                     '#7F8C8D'),
    ]
    shapes, ann = [], []
    for cx,cy,lbl,col in nodes:
        shapes.append(dict(type='rect', x0=cx-BW/2, y0=cy-BH/2,
                           x1=cx+BW/2, y1=cy+BH/2,
                           fillcolor=col, opacity=0.88,
                           line=dict(color='white', width=1.5)))
        ann.append(dict(x=cx,y=cy,text=lbl,showarrow=False,
                        font=dict(color='white',size=10),align='center'))
    edges = [(0,1,''),(0,4,''),(1,2,''),(2,3,''),(4,5,''),(5,6,''),
             (3,7,'Yes'),(3,9,'No'),(6,8,'Yes'),(6,9,'No')]
    for si,di,lbl in edges:
        sx,sy=nodes[si][0],nodes[si][1]; dx,dy=nodes[di][0],nodes[di][1]
        ann.append(dict(x=dx,y=dy+BH/2+.005,ax=sx,ay=sy-BH/2-.005,
                        xref='x',yref='y',axref='x',ayref='y',
                        showarrow=True,arrowhead=2,arrowsize=1.2,arrowwidth=1.5,
                        arrowcolor='#555',text=lbl,font=dict(size=9,color='#333')))
    fig.update_layout(shapes=shapes,annotations=ann,
                      plot_bgcolor='#FAFAFA',paper_bgcolor='white',
                      margin=dict(l=10,r=10,t=40,b=10),height=580,
                      title=dict(text='Signal -> Strategy Architecture',x=0.5,font=dict(size=15)))
    fig.add_trace(go.Scatter(x=[0,1],y=[0,1],mode='markers',
                             marker=dict(opacity=0),showlegend=False))
    fig.show()
arch()

---
## Theory Base — Books to Code

In [ ]:
theory = pd.DataFrame([
    ['Natenberg — *Option Volatility & Pricing* Ch.7',
     'Theta budget gate: reject entry when |daily theta| > 1.5% of premium',
     'options_backtester._option_theta(), entry filter theta_pct <= 0.015'],
    ['Natenberg — Ch.7 / Ch.8',
     'Full Greeks suite: delta, gamma, theta, vega on GreeksEstimator',
     'options_trader.GreeksEstimator.{delta,gamma,theta,vega}'],
    ['Sinclair — *Volatility Trading* Ch.2',
     'IV/RV ratio as core edge metric; filter entries when IV >> RV',
     'options_backtester._iv_rv_ratio(), IVEstimator.iv_rv_ratio()'],
    ['Sinclair — Ch.3',
     'Vega exposure: enter straddles at low IVR so long vega is cheap',
     'IVR <= 25 gate + vol_prob >= 0.30 replaces hard binary vol_expanding'],
    ['Sinclair — Ch.8 / Davey — *Building Winning Systems*',
     'Quarter-Kelly sizing: f = (p - (1-p)/b) * 0.25',
     'options_backtester._kelly_fraction()'],
    ['Davey',
     'tradeable meta-RF used as Kelly boost, not hard veto',
     'confidence_threshold lowered; kelly_scale = 0.5 + 0.5*kelly'],
    ['McMillan — *Options as a Strategic Investment*',
     'Close losing straddle leg when winning leg >= 3x loser',
     'options_backtester._manage_straddle() call_val >= 3*put_val rule'],
], columns=['Source', 'Principle', 'Implementation'])

theory.style.set_properties(**{'text-align':'left'}).hide(axis='index')

---
## Greeks Profile at Entry
Computed for representative entries at current VIX=20.  
Directional: ITM call, delta~0.68, 28 DTE.  Straddle: ATM call+put, 30 DTE.

In [ ]:
IV_SCALE = {'SPY':1.0,'QQQ':1.1,'IWM':1.15,'SOXX':1.2,
            'GLD':0.75,'SLV':1.2,'XLE':1.25,'EWT':1.1,
            'EWS':1.1,'EEM':1.15,'EWJ':1.0,'INDA':1.2}

rows = []
test_prices = {'SPY':580,'QQQ':500,'IWM':210,'GLD':240,'SLV':30,'XLE':90}
VIX = 20.0

for sym, S in test_prices.items():
    sigma = max(0.10, (VIX/100) * IV_SCALE.get(sym, 1.0))
    # ── Directional call (ITM ~delta 0.68) ──
    T_d = 28/365
    step = 5.0 if S >= 100 else 1.0
    best_k, best_diff = S, 999
    for k_off in range(-60, 5, int(step)):
        k = round(S/step)*step + k_off
        if k <= 0: continue
        diff = abs(bs_delta(True, S, k, T_d, sigma) - 0.68)
        if diff < best_diff: best_diff, best_k = diff, k
    K_call = best_k
    px_call = bs_price(True, S, K_call, T_d, sigma)
    theta_pct = abs(bs_theta(True, S, K_call, T_d, sigma)) * 100 / (px_call*100) if px_call>0 else 0
    rows.append({'Symbol': sym, 'Strategy': 'Directional CALL',
                 'Strike': K_call, 'DTE': 28, 'sigma': round(sigma,3),
                 'Price': round(px_call*100,1),
                 'Delta': round(bs_delta(True, S, K_call, T_d, sigma),3),
                 'Gamma': round(bs_gamma(S, K_call, T_d, sigma),4),
                 'Theta/day': round(bs_theta(True, S, K_call, T_d, sigma)*100,2),
                 'Vega/1%': round(bs_vega(S, K_call, T_d, sigma)*100,2),
                 'theta% cost': round(theta_pct*100,2)})
    # ── Straddle (ATM) ──
    T_s = 30/365
    K_atm = round(S/5)*5 if S >= 100 else round(S)
    px_c = bs_price(True,  S, K_atm, T_s, sigma)
    px_p = bs_price(False, S, K_atm, T_s, sigma)
    total = (px_c + px_p)*100
    rows.append({'Symbol': sym, 'Strategy': 'Straddle',
                 'Strike': K_atm, 'DTE': 30, 'sigma': round(sigma,3),
                 'Price': round(total,1),
                 'Delta': round(bs_delta(True,S,K_atm,T_s,sigma)+bs_delta(False,S,K_atm,T_s,sigma),3),
                 'Gamma': round(2*bs_gamma(S, K_atm, T_s, sigma),4),
                 'Theta/day': round((bs_theta(True,S,K_atm,T_s,sigma)+bs_theta(False,S,K_atm,T_s,sigma))*100,2),
                 'Vega/1%': round(2*bs_vega(S,K_atm,T_s,sigma)*100,2),
                 'theta% cost': round(abs((bs_theta(True,S,K_atm,T_s,sigma)+bs_theta(False,S,K_atm,T_s,sigma))*100/total)*100,2) if total>0 else 0})

greeks_df = pd.DataFrame(rows)

def hl(row):
    bg = '#E3F2FD' if row['Strategy']=='Directional CALL' else '#FFF3E0'
    ok = '#FFCDD2' if row['theta% cost'] > 1.5 else bg  # red if theta too high
    return [bg]*9 + [ok]

greeks_df.style.apply(hl, axis=1).format({
    'Price':'${:.0f}','Theta/day':'${:.2f}',
    'Vega/1%':'${:.2f}','theta% cost':'{:.2f}%'
}).set_properties(**{'text-align':'right'}).hide(axis='index')

---
## Kelly Position Sizing
`f = (p - (1-p)/b) * 0.25`  — Quarter-Kelly (Sinclair Ch.8 / Davey).  
Kelly scale maps to `0.50 – 0.625 × max_risk` so minimum sizing is always 50%.

In [ ]:
probs   = np.linspace(0.10, 0.95, 200)
b_dir   = 0.50 / 0.25    # directional: profit/loss = 2.0
b_strad = 0.80 / 0.40    # straddle:    profit/loss = 2.0

def kelly(p, b):
    raw = p - (1-p)/b
    return max(0.0, min(raw, 1.0)) * 0.25

kelly_dir   = [0.5 + 0.5*kelly(p, b_dir)   for p in probs]
kelly_strad = [0.5 + 0.5*kelly(p, b_strad) for p in probs]

fig = go.Figure()
fig.add_trace(go.Scatter(x=probs, y=kelly_dir,   name='Directional (b=2.0)',
    line=dict(color='#2196F3', width=2)))
fig.add_trace(go.Scatter(x=probs, y=kelly_strad, name='Straddle (b=2.0)',
    line=dict(color='#FF9800', width=2, dash='dash')))
fig.add_hline(y=0.50, line_dash='dot', line_color='gray',
    annotation_text='50% floor', annotation_position='bottom right')
fig.add_vline(x=0.65, line_dash='dot', line_color='#27AE60',
    annotation_text='dir entry threshold', annotation_position='top right')
fig.update_layout(title=dict(text='Quarter-Kelly Scale Factor vs Confidence',x=0.5),
    xaxis_title='Confidence / vol_prob', yaxis_title='Risk scale (fraction of max_risk)',
    yaxis=dict(tickformat='.0%', range=[0.45, 0.68]),
    height=380, template='plotly_white')
fig.show()

---
## Strategy Comparison

In [ ]:
strat = pd.DataFrame({
    'Parameter': ['Signal source','Entry confidence','IV Rank max',
                  'IV/RV max','Theta budget','Kelly sizing',
                  'Strike','Expiry','Profit target','Stop loss',
                  'Force close','Extra exits'],
    'Directional ITM': [
        'DirectionLSTM conf','>= 0.65 (+ Kelly boost if tradeable)','IVR <= 55',
        '<= 1.75','|theta|/premium <= 1.5%','0.25-Kelly on confidence',
        'Delta ~0.68 ITM via BS d1','28 DTE','+50% (1.5x)','-25% (0.75x)',
        '7 DTE','Direction flip (conf >= 0.50)'],
    'Vol Expansion Straddle': [
        'VolExpansionLSTM prob','vol_prob >= 0.30 + dir conf < 0.45','IVR <= 25',
        '<= 1.70','none','0.25-Kelly on vol_prob',
        'ATM = round(price)','30 DTE','+80% on either leg (1.8x)','-40% total (0.6x)',
        '5 DTE','McMillan 3x rule — close losing leg when winner >= 3x loser'],
}).set_index('Parameter')

def cs(s):
    return (['background-color:#E3F2FD']*len(s) if s.name=='Directional ITM'
            else ['background-color:#FFF3E0']*len(s))
strat.style.apply(cs).set_properties(**{'text-align':'left'})

In [ ]:
cats  = ['Profit Target%','Stop Loss%','Force Close DTE',
         'Expiry DTE','Min Conf%','Max IVR','Max IV/RV x10']
dir_v = [50, 25, 7, 28, 65, 55, 17]
str_v = [80, 40, 5, 30, 30, 25, 17]

fig = go.Figure()
for vals, name, col in [(dir_v,'Directional ITM','#2196F3'),(str_v,'Straddle','#FF9800')]:
    fig.add_trace(go.Scatterpolar(r=vals+[vals[0]], theta=cats+[cats[0]],
        fill='toself', name=name, line=dict(color=col)))
fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0,85])),
    title=dict(text='Strategy Parameters Radar',x=0.5),
    height=420, template='plotly_white')
fig.show()

---
## IV/RV Edge (Sinclair Ch.2)
`ratio = (VIX/100 × sym_scale) / realized_vol_20d`  
- `< 1.0` → options cheap vs. recent moves — buying edge  
- `> 1.25` → options expensive — historical equity VRP  
- `> 1.75` → blocks directional entry &nbsp; `> 1.70` → blocks straddle entry

In [ ]:
# Load backtest data to compute historical IV/RV snapshots
eq_files = sorted(OUTPUTS.glob('backtest_*.csv'))
if eq_files:
    # Build a synthetic IV/RV distribution using IV_SCALE and a fixed RV range
    rvs    = np.linspace(0.08, 0.45, 300)
    vix_lo, vix_hi = 12.0, 40.0
    syms_show = ['SPY','QQQ','IWM','GLD','SLV','XLE']

    fig = go.Figure()
    for sym in syms_show:
        scale = IV_SCALE.get(sym, 1.0)
        vix_mid = (vix_lo + vix_hi) / 2
        iv = max(0.10, (vix_mid/100)*scale)
        ratios = iv / rvs
        fig.add_trace(go.Scatter(x=rvs, y=ratios, mode='lines', name=sym))

    fig.add_hline(y=1.0,  line_dash='dot', line_color='green',
        annotation_text='edge threshold (1.0)', annotation_position='bottom right')
    fig.add_hline(y=1.25, line_dash='dot', line_color='orange',
        annotation_text='VRP norm (1.25)', annotation_position='bottom right')
    fig.add_hline(y=1.75, line_dash='dash', line_color='red',
        annotation_text='directional block (1.75)', annotation_position='top right')
    fig.update_layout(
        title=dict(text='IV/RV Ratio vs Realized Vol  (VIX=26 midpoint)', x=0.5),
        xaxis_title='20-day Realized Vol', yaxis_title='IV / RV',
        height=380, template='plotly_white')
    fig.show()

---
## Vol Feature Columns (10 features)

In [ ]:
vf = pd.DataFrame({
    'Feature':['vol20','bb_bandwidth','vol_regime','bb_pct_b','ret5',
               'rsi14','vix','vix_chg','momentum_quality','adx'],
    'Category':['Volatility']*4 + ['Momentum']*2 + ['Macro']*2 + ['Momentum','Trend'],
    'Role':['Primary expansion target','Current vol regime','High/low vol classification',
            'Position within BB bands','5-bar return','RSI(14)',
            'VIX level','VIX daily change','Momentum quality score','ADX trend strength'],
})
cat_col = {'Volatility':'#8E44AD','Momentum':'#2196F3','Macro':'#E67E22','Trend':'#27AE60'}

counts = vf['Category'].value_counts()
fig = go.Figure(go.Bar(
    y=counts.index, x=counts.values, orientation='h',
    marker_color=[cat_col[c] for c in counts.index],
    text=counts.values, textposition='auto'))
fig.update_layout(title=dict(text='Vol Feature Categories',x=0.5),
    height=280, template='plotly_white', margin=dict(l=100))
fig.show()

def hl_cat(row):
    c = cat_col.get(row['Category'],'#ccc')
    return [f'border-left:4px solid {c}']*len(row)
vf.style.apply(hl_cat,axis=1).set_properties(**{'text-align':'left'}).hide(axis='index')

---
## Model Inventory

In [ ]:
info_path = ROOT / 'models_info.json'
if info_path.exists():
    info = json.loads(info_path.read_text())
    inv  = pd.DataFrame([{'Symbol':m['symbol'],'Mode':m['mode'],
                           'Features':m['features'],'KB':m['size_kb'],
                           'Status':m['status'],
                           'Trained':m['trained_at'][:16].replace('T',' '),
                           'Note':m.get('note','')} for m in info['models']])
    sc = {'active':'#C8E6C9','benched':'#FFF9C4','excluded':'#FFCDD2'}
    def cs2(row): bg=sc.get(row['Status'],'#fff'); return [f'background-color:{bg}']*len(row)
    display(Markdown(f"**{info['total']} models** — generated {info['generated_at'][:16]}"))
    inv.style.apply(cs2,axis=1).set_properties(**{'text-align':'left'}).hide(axis='index')
else:
    display(Markdown('`models_info.json` not found.'))

In [ ]:
if info_path.exists():
    status = inv.groupby('Status').agg(Count=('Symbol','size'),
        Symbols=('Symbol',lambda x:', '.join(sorted(x.unique())))).reset_index()
    fig = go.Figure(go.Pie(
        labels=status['Status'], values=status['Count'],
        marker=dict(colors=[sc.get(s,'#ccc') for s in status['Status']]),
        textinfo='label+value', hole=0.4))
    fig.update_layout(title=dict(text='Model Status',x=0.5), height=300,
                      template='plotly_white')
    fig.show()

---
## Backtest Equity Curves

In [ ]:
eq_files = sorted(OUTPUTS.glob('backtest_*.csv'))
final_eq = {}
if eq_files:
    fig = go.Figure()
    for f in eq_files:
        sym = f.stem.replace('backtest_','')
        df  = pd.read_csv(f, parse_dates=['date'])
        df  = df.groupby('date')['equity'].last().reset_index()
        final_eq[sym] = df['equity'].iloc[-1]
        fig.add_trace(go.Scatter(x=df['date'],y=df['equity'],mode='lines',name=sym))
    fig.add_hline(y=100_000, line_dash='dash', line_color='gray',
                  annotation_text='$100k start')
    fig.update_layout(title=dict(text='Equity Curves',x=0.5),
        xaxis_title='Date', yaxis_title='Equity ($)', yaxis=dict(tickformat='$,.0f'),
        height=480, template='plotly_white', hovermode='x unified')
    fig.show()

In [ ]:
if final_eq:
    fe = pd.Series(final_eq).sort_values()
    fig = go.Figure(go.Bar(
        y=fe.index, x=fe.values-100_000, orientation='h',
        marker_color=['#C0392B' if v<100_000 else '#27AE60' for v in fe.values],
        text=[f'${v-100_000:+,.0f}' for v in fe.values], textposition='auto'))
    fig.add_vline(x=0, line_color='gray', line_dash='dash')
    fig.update_layout(title=dict(text='Total P&L vs $100k Start',x=0.5),
        xaxis=dict(tickformat='$,.0f'), height=500, template='plotly_white')
    fig.show()

---
## Trade Analysis

In [ ]:
t_files = sorted(OUTPUTS.glob('trades_*.csv'))
all_t   = []
if t_files:
    for f in t_files:
        df = pd.read_csv(f)
        if not df.empty:
            df['symbol'] = f.stem.replace('trades_','')
            all_t.append(df)

if all_t:
    trades = pd.concat(all_t, ignore_index=True)
    summ = trades.groupby('symbol').agg(
        Trades=('pnl','size'), Wins=('pnl',lambda x:(x>0).sum()),
        Total=('pnl','sum'), Avg=('pnl','mean'),
        Best=('pnl','max'), Worst=('pnl','min')
    ).reset_index()
    summ['WR%']  = (summ['Wins']/summ['Trades']*100).round(1)
    summ = summ.sort_values('Total', ascending=False)
    display(Markdown(f'**{len(trades)} trades across {trades["symbol"].nunique()} symbols**'))
    def cg(row): bg='#E8F5E9' if row['Total']>0 else '#FFEBEE'; return [f'background-color:{bg}']*len(row)
    summ.style.apply(cg,axis=1).format({
        'Total':'${:,.0f}','Avg':'${:,.0f}','Best':'${:,.0f}',
        'Worst':'${:,.0f}','WR%':'{:.1f}%'
    }).set_properties(**{'text-align':'right'}).hide(axis='index')

In [ ]:
if all_t:
    sub = make_subplots(rows=1,cols=2, subplot_titles=('P&L Distribution','Exit Reasons'))
    for sym, grp in trades.groupby('symbol'):
        sub.add_trace(go.Histogram(x=grp['pnl'], name=sym, opacity=0.6,
                                   nbinsx=30), row=1, col=1)
    if 'exit_reason' in trades.columns:
        er = trades.groupby('exit_reason').agg(Count=('pnl','size'),
                                                Avg=('pnl','mean')).sort_values('Count',ascending=False)
        sub.add_trace(go.Bar(x=er.index, y=er['Count'],
            marker_color='#2196F3', showlegend=False), row=1, col=2)
    sub.add_vline(x=0, line_color='red', line_dash='dash', row=1, col=1)
    sub.update_layout(height=380, template='plotly_white', barmode='overlay', showlegend=False)
    sub.show()

---
## Active Symbols

In [ ]:
groups = {'Intraday':['SPY','QQQ','IWM','SOXX'],
          'Swing':['EWT','GLD','EEM','SLV'],
          'Expansion':['EWJ','EWS','XLE'],
          'Excluded':['TLT','IGV','FXI']}
gc = {'Intraday':'#BBDEFB','Swing':'#C8E6C9','Expansion':'#FFF9C4','Excluded':'#FFCDD2'}

sym_df = pd.DataFrame([{'Group':g,'Symbol':s} for g,ss in groups.items() for s in ss])
display(Markdown('> TLT / IGV / FXI excluded — poor directional win rate.'))
sym_df.style.apply(
    lambda row:[f'background-color:{gc.get(row["Group"],"#fff")}']*len(row),axis=1
).set_properties(**{'text-align':'left'}).hide(axis='index')